# Similarity search — TF-IDF + exact cosine & LSH

Interactive companion to `similarity_search.py` / `run_smoke.py`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from similarity_search import (
    DOCUMENTS, LABELED_QUERIES, RandomProjectionLSH,
    build_vocab, compute_idf, tfidf_matrix, l2_normalize,
    embed_query, exact_topk, recall_at_k, topic_of,
)
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
word2id, id2word = build_vocab(DOCUMENTS)
idf = compute_idf(DOCUMENTS, word2id)
X = l2_normalize(tfidf_matrix(DOCUMENTS, word2id, idf))
print(f'docs={len(DOCUMENTS)} vocab={len(id2word)} dim={X.shape[1]}')
for d in DOCUMENTS[:3]:
    print(d['id'], d['topic'], ':', d['text'][:60], '...')


In [ ]:
q = 'python java programming code'
qv = embed_query(q, word2id, idf)
idx, scores = exact_topk(qv, X, k=5)
print('Exact top-5 for:', q)
for i, s in zip(idx, scores):
    print(f'  {DOCUMENTS[int(i)]["id"]} ({topic_of(int(i))}) cos={s:.3f}  {DOCUMENTS[int(i)]["text"][:55]}')

plt.figure(figsize=(7,3.5))
plt.bar(range(len(idx)), scores)
plt.xticks(range(len(idx)), [DOCUMENTS[int(i)]['id'] for i in idx])
plt.ylabel('cosine'); plt.title('Exact top-5'); plt.ylim(0,1.05); plt.grid(True, axis='y', alpha=0.3)
plt.show()


In [ ]:
lsh = RandomProjectionLSH(dim=X.shape[1], n_bits=64, seed=42).fit(X)
approx_idx, ham = lsh.query(qv, k=5)
rr_idx, rr_scores = lsh.query_candidates(qv, candidate_pool=8, doc_matrix=X, k=5)
print('LSH Hamming top-5:', [DOCUMENTS[int(i)]['id'] for i in approx_idx], 'hamming', ham.tolist())
print('LSH+rerank top-5:', [DOCUMENTS[int(i)]['id'] for i in rr_idx], 'cos', [round(float(s),3) for s in rr_scores])
print('recall@5 Hamming', recall_at_k(idx, approx_idx))
print('recall@5 rerank ', recall_at_k(idx, rr_idx))


In [ ]:
for item in LABELED_QUERIES:
    qv = embed_query(item['query'], word2id, idf)
    idx, scores = exact_topk(qv, X, k=3)
    topics = [topic_of(int(i)) for i in idx]
    ok = item['expected_topic'] in topics
    print(('PASS' if ok else 'FAIL'), item['query'][:35], '->', list(zip([DOCUMENTS[int(i)]['id'] for i in idx], topics)))
